# 06 · Week 3 prediction, fusion và robustness (Kaggle)

Notebook này không train lại mặc định. Nó load prediction/checkpoint artifacts, chạy corruption inference cho R/C và tạo mean-probability fusion.
Không retune threshold hoặc checkpoint trên corruption.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path('/kaggle/working/FaceTrace')
if not (REPO_DIR / 'src').is_dir():
    raise FileNotFoundError('Attach hoặc clone đúng snapshot FaceTrace trước khi chạy.')
sys.path.insert(0, str(REPO_DIR))

# Chỉnh các đường dẫn theo Kaggle Input version đã attach.
WEEK3_ROOT = Path('/kaggle/input/week3-artifacts')
DATA_ROOT = Path('/kaggle/input/who-is-ai/data')
VAL_MANIFEST = WEEK3_ROOT / 'splits/val.csv'
ANALYSIS_DIR = Path('/kaggle/working/week3/analysis')
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Ví dụ: chạy residual checkpoint trên clean/JPEG/resize/blur.
# Sửa CHECKPOINT và OUTPUT_ROOT nếu artifact layout khác.
CHECKPOINT = WEEK3_ROOT / 'seed42/residual_only/best_checkpoint.pt'
OUTPUT_ROOT = Path('/kaggle/working/week3/seed42/residual_only')
for condition in ('clean', 'jpeg70', 'resize112', 'blur1'):
    subprocess.run([
        sys.executable, str(REPO_DIR / 'scripts/predict_residual.py'),
        '--checkpoint', str(CHECKPOINT),
        '--manifest', str(VAL_MANIFEST),
        '--image-path-base', str(DATA_ROOT.parent.parent),
        '--output', str(OUTPUT_ROOT / f'validation_predictions_{condition}.csv'),
        '--corruption', condition,
    ], cwd=REPO_DIR, check=True)

In [ ]:
# Clean fusion. Cần attach prediction CSV của Global/Local từ Phase 5/6 với cùng image_id.
# Ví dụ CLI (bỏ các branch không có artifact):
# subprocess.run([
#   sys.executable, str(REPO_DIR / 'scripts/evaluate_week3.py'),
#   '--prediction', f'global={WEEK3_ROOT}/seed42/global_only/validation_predictions.csv',
#   '--prediction', f'local={WEEK3_ROOT}/seed42/local_only/validation_predictions.csv',
#   '--prediction', f'residual={OUTPUT_ROOT}/validation_predictions_clean.csv',
#   '--output-dir', str(ANALYSIS_DIR / 'clean_seed42'),
# ], cwd=REPO_DIR, check=True)

In [ ]:
# Không xem metric corruption là test độc lập; dùng checkpoint/threshold clean đã khóa.
for path in sorted((Path('/kaggle/working/week3')).rglob('*.metrics.json')):
    print(path)